In [23]:
import pandas as pd
import db_dtypes
import scipy.stats
from statsmodels.stats.multitest import multipletests
import sys
sys.path.append('../src')
from stats import week_over_week, adjust_pvalues

In [2]:
data_df = pd.read_parquet('../data/cache/02_sessions.parquet', engine='pyarrow')
data_df.head(5)

,user_pseudo_id,session_id,visit_start,device_category,n_events,n_devices,country,n_countries,source_medium,has_view_item,has_add_to_cart,has_begin_checkout,has_add_shipping_info,has_add_payment_info,has_purchase,medium_group,visit_start_date
0,1034159.5088169437,598302469,2021-01-10 04:41:38.049786+00:00,mobile,35,1,Italy,1,<Other> / <Other>,True,False,False,False,False,False,Unknown,2021-01-10
1,1051092.0412766822,5382659029,2021-01-10 17:25:05.792659+00:00,mobile,4,1,Brazil,1,NaN,False,False,False,False,False,False,Direct,2021-01-10
2,1068310.8866508729,2875162679,2021-01-10 05:07:25.629800+00:00,mobile,11,1,United States,1,google / organic,True,False,False,False,False,False,Organic Search,2021-01-10
3,1120618.5114122220,3096417581,2021-01-10 14:34:49.632279+00:00,desktop,5,1,France,1,NaN,False,False,False,False,False,False,Direct,2021-01-10
4,1130929.6658219391,938613650,2021-01-10 00:24:51.885166+00:00,mobile,6,1,United States,1,google / organic,False,False,False,False,False,False,Organic Search,2021-01-10


In [3]:
total_events = data_df['n_events'].sum()

print(f"Number of rows in data_df: {len(data_df)}")
print(f"Total events in data_df: {total_events}")


Number of rows in data_df: 360129
Total events in data_df: 4295584


In [4]:
data_df['visit_start_date'] = pd.to_datetime(data_df['visit_start_date'])
data_df['week'] = data_df['visit_start_date'].dt.to_period("W-SUN").dt.start_time
start_date = '2020-11-02'
end_date = '2021-01-31'
filtered_df = data_df[data_df['visit_start_date'].between(start_date, end_date)]
print(f"Check on filtered data: len(filtered_df) is {len(filtered_df)} and should be 357,504")
print(f"Check on filtered data: the number of Mondays is {filtered_df['week'].nunique()} and should be 13")


Check on filtered data: len(filtered_df) is 357504 and should be 357,504
Check on filtered data: the number of Mondays is 13 and should be 13


In [5]:

weekly = filtered_df.groupby('week').agg(
   visits=('user_pseudo_id', 'size'),
    purchasing_visits= ('has_purchase', 'sum')
)
weekly['conversion']= weekly['purchasing_visits'] / weekly['visits']


In [6]:

phi_visits = (((weekly['visits']-weekly['visits'].shift(1))**2)/(weekly['visits']+weekly['visits'].shift(1))).mean()
phi_purchases = (((weekly['purchasing_visits']-weekly['purchasing_visits'].shift(1))**2)/(weekly['purchasing_visits']+weekly['purchasing_visits'].shift(1))).mean()
print(f"Dispersion of weekly visits (Poisson = 1) is : {phi_visits:.2f}")
print(f"Dispersion of weekly purchasing visits (Poisson = 1) is: {phi_purchases:.2f}")


Dispersion of weekly visits (Poisson = 1) is : 480.79
Dispersion of weekly purchasing visits (Poisson = 1) is: 26.05


In [7]:
def count_change_test(last_week_count, this_week_count, phi = 1):
    lwc = last_week_count
    twc = this_week_count
    if lwc == 0 and twc == 0:
        return 0.0, 1.0
    z = (twc - lwc) / ((phi * (twc + lwc)) ** 0.5)
    p = 2 * scipy.stats.norm.sf(abs(z))
    return float(z), float(p)

In [8]:
count_change_test(772, 549)

(-6.1355493588510726, 8.486541039552783e-10)

In [9]:
count_change_test(772, 549, phi=phi_purchases)

(-1.2020452239945862, 0.22934600696371943)

In [10]:
def two_proportion_test(last_week_purchases, last_week_visits, this_week_purchases, this_week_visits):

    lwp = last_week_purchases
    twp = this_week_purchases
    lv = last_week_visits
    tv = this_week_visits
    if lv == 0 or tv == 0:
        return 0.0, 1.0       
    pr = (lwp+twp)/(lv+tv)
    if pr == 0 or pr == 1:
        return 0.0, 1.0

    z = (twp/tv - lwp/lv)/ ((pr*(1-pr)*(1/tv + 1/lv))**0.5)
    p = 2 * scipy.stats.norm.sf(abs(z))
    return float(z), float(p) 

In [11]:
two_proportion_test(772, 38729, 549, 33411)

(-3.4978385006057953, 0.0004690450677550937)

In [12]:
def coverage_guard(timestamp, uses_cart = False):

    if timestamp.dayofweek !=0:
        raise ValueError(f"{timestamp.date()}: The day is not Monday")
    elif timestamp < pd.Timestamp('2020-11-02') or timestamp > pd.Timestamp('2021-01-25'):
        raise ValueError(f"{timestamp.date()}: The whole week is not inside the data")
    elif uses_cart and timestamp < pd.Timestamp('2020-11-30'):
        raise ValueError(f"{timestamp.date()}: add_to_cart is not fully tracked before 2020-11-30")

In [13]:
coverage_guard(pd.Timestamp("2020-12-07"), uses_cart=True)

In [14]:
try:
    coverage_guard(pd.Timestamp("2020-11-23"), uses_cart=True)
except ValueError as e:
    print("Refused, as expected:", e)    

Refused, as expected: 2020-11-23: add_to_cart is not fully tracked before 2020-11-30


In [15]:
try:
    coverage_guard(pd.Timestamp("2020-12-08"))
except ValueError as e:
    print("Refused, as expected:", e)    

Refused, as expected: 2020-12-08: The day is not Monday


In [16]:
def adjust_pvalues(pvals):
    """
    Adjust p-values for multiple testing using the Benjamini-Hochberg method.
    
    Parameters:
    pvals (list or array-like): List of p-values to adjust.
    
    Returns:
    list: Adjusted p-values.
    """
    if len(pvals) == 0:
        return []
    
    _, adj_pvals, _, _ = multipletests(pvals, method='fdr_bh')

    return adj_pvals.tolist()

In [17]:
adjust_pvalues([0.001, 0.04, 0.5])

[0.003, 0.060000000000000005, 0.5]

In [18]:
weekly.head(5)

,visits,purchasing_visits,conversion
week,,,
2020-11-02,24991,249,0.009964
2020-11-09,22508,341,0.01515
2020-11-16,25914,374,0.014432
2020-11-23,27780,515,0.018539
2020-11-30,31088,590,0.018978


In [ ]:
rows = []
for i in range(1,13):
    lw = weekly.iloc[i-1]
    tw = weekly.iloc[i]
    label = weekly.index[i]
    coverage_guard(label, uses_cart=False)
    coverage_guard(weekly.index[i-1], uses_cart=False)

    z, p = count_change_test(lw['visits'], tw['visits'], phi=phi_visits)
    visits_dict = {"week": label, "metric": "visits", "last": lw['visits'], "this": tw['visits'], "z": z, "p": p}
    rows.append(visits_dict)
    z, p = count_change_test(lw['purchasing_visits'], tw['purchasing_visits'], phi=phi_purchases)
    purchasing_visits_dict = {"week": label, "metric": "purchasing_visits", "last": lw['purchasing_visits'], "this": tw['purchasing_visits'], "z": z, "p": p}
    rows.append(purchasing_visits_dict)

    z, p = two_proportion_test(lw['purchasing_visits'], lw['visits'], tw['purchasing_visits'], tw['visits'])
    conversion_dict = {"week": label, "metric": "conversion", "last": lw['conversion'], "this": tw['conversion'], "z": z, "p": p}
    rows.append(conversion_dict)

results = pd.DataFrame(rows)
results['p_adj'] = adjust_pvalues(results['p'])
results['flag'] = results['p_adj'] < 0.05

results2 = week_over_week(weekly, phi_visits, phi_purchases)
results2['p_adj'] = adjust_pvalues(results2['p'])
results2['flag'] = results2['p_adj'] < 0.05



In [27]:
pd.testing.assert_frame_equal(results, results2)